# Tahoe Regional Transportation Plan - Scenario Forecast Template

In [ ]:
# import packages
import pandas as pd
import pathlib
from pathlib import Path
import os
import arcpy
from utils import *
import numpy as np
import pickle
# external connection packages
from sqlalchemy.engine import URL
from sqlalchemy import create_engine

# pandas options
pd.options.mode.copy_on_write = True
pd.options.mode.chained_assignment = None
pd.options.display.max_columns = 999
pd.options.display.max_rows    = 999

# my workspace 
workspace = r"C:\GIS\Scratch.gdb"
# current working directory
local_path = pathlib.Path().absolute()

# get bonus_condit
# set data path as a subfolder of the current working directory TravelDemandModel\2022\
data_dir = local_path.parents[0] / 'data'
# folder to save processed data
out_dir  = local_path.parents[0] / 'data/processed_data'
# workspace gdb for stuff that doesnt work in memory
# gdb = os.path.join(local_path,'Workspace.gdb')
gdb = workspace
# set environement workspace to in memory 
arcpy.env.workspace = 'memory'
# # clear memory workspace
# arcpy.management.Delete('memory')

# overwrite true
arcpy.env.overwriteOutput = True
# Set spatial reference to NAD 1983 UTM Zone 10N
sr = arcpy.SpatialReference(26910)

# get parcels from the database
# network path to connection files
filePath = "F:/GIS/PARCELUPDATE/Workspace/"
# database file path 
sdeBase    = os.path.join(filePath, "Vector.sde")
sdeCollect = os.path.join(filePath, "Collection.sde")
sdeTabular = os.path.join(filePath, "Tabular.sde")
sdeEdit    = os.path.join(filePath, "Edit.sde")

# Pickle variables
# part 1 - spatial joins and new categorical fields
parcel_pickle_part1    = data_dir / 'parcel_pickle1.pkl'
# part 2 - forecasting applied
parcel_pickle_part2    = data_dir / 'parcel_pickle2.pkl'

# columsn to list
initial_columns = [ 'APN',
                    'APO_ADDRESS',
                    'Residential_Units',
                    'TouristAccommodation_Units',
                    'CommercialFloorArea_SqFt',
                    'YEAR',
                    'JURISDICTION',
                    'COUNTY',
                    'OWNERSHIP_TYPE',
                    'COUNTY_LANDUSE_DESCRIPTION',
                    'EXISTING_LANDUSE',
                    'REGIONAL_LANDUSE',
                    'YEAR_BUILT',
                    'PLAN_ID',
                    'PLAN_NAME',
                    'ZONING_ID',
                    'ZONING_DESCRIPTION',
                    'TOWN_CENTER',
                    'LOCATION_TO_TOWNCENTER',
                    'TAZ',
                    'PARCEL_ACRES',
                    'PARCEL_SQFT',
                    'WITHIN_BONUSUNIT_BNDY',
                    'WITHIN_TRPA_BNDY',
                    'SHAPE']

In [ ]:
# Load base parcel data exported by parcel_engineering
sdfParcels = pd.read_pickle(out_dir / 'base_parcel_data.pkl')

# Load lookup lists

res_zoned_lookup      = "Lookup_Lists/forecast_residential_zoned_units.csv"

dfResZoned    = pd.read_csv(res_zoned_lookup)


> Subtract Assigned Units from the appropriate pool

# Start of scenarios

> Setup Unit Pools Proportion to be Used in Each Zone

In [ ]:
dfPool = pd.read_csv(res_zoned_lookup)
# proportion target of forecast development by type
portion_multifamily = .35
portion_singlefamily = .5
portion_infill = .15
# Set units to use for each zoning type
dfPool['Future_Units_Adjusted'] = dfPool['Future_Units_Adjusted'].fillna(0)
dfPool['Future_Units_Adjusted_MF'] = (dfPool['Future_Units_Adjusted'] * portion_multifamily).round().astype(int)
dfPool['Future_Units_Adjusted_SF'] = (dfPool['Future_Units_Adjusted'] * portion_singlefamily).round().astype(int)
dfPool['Future_Units_Adjusted_Infill'] = (dfPool['Future_Units_Adjusted'] * portion_infill).round().astype(int)
# Assign any rounding error to the single family pool
dfPool['Adjustment'] = dfPool['Future_Units_Adjusted'] - dfPool['Future_Units_Adjusted_MF'] - dfPool['Future_Units_Adjusted_SF'] - dfPool['Future_Units_Adjusted_Infill']
dfPool['Future_Units_Adjusted_SF'] = dfPool['Future_Units_Adjusted_SF'] + dfPool['Adjustment']
dfPool.drop(columns=['Adjustment'], inplace=True)

> Define Conditions

In [ ]:
conditions = get_parcel_conditions()

In [ ]:
# check parcel conditions
df_potential = pd.DataFrame()
for key,value in conditions.items():
    df = check_parcel_condition(sdfParcels, value)
    df['Condition'] = key
    df_potential = pd.concat([df_potential, df], axis=0)

df_potential

> Forecast Jurisdiction Pools

In [ ]:
conditions = get_parcel_conditions()

In [ ]:
## Jurisdictional Forecasting ##
# (condition_key, jurisdiction, pool, type, label, forecast_function)
assignments = [
    ##-- Bonus Unit Assignments --##
    ('CSLT_Bonus_MF',      'CSLT', 'Bonus Unit', 'MF',     'CSLT Bonus Units MF',     forecast_residential_units),
    ('CSLT_Bonus_SF',      'CSLT', 'Bonus Unit', 'SF',     'CSLT Bonus Units SF',     forecast_residential_units),
    ('CSLT_Bonus_Infill',  'CSLT', 'Bonus Unit', 'Infill', 'CSLT Bonus Units Infill', forecast_residential_units_infill),
    ('DG_Bonus_MF',        'DG',   'Bonus Unit', 'MF',     'DG Bonus Units MF',       forecast_residential_units),
    ('DG_Bonus_SF',        'DG',   'Bonus Unit', 'SF',     'DG Bonus Units SF',       forecast_residential_units),
    ('DG_Bonus_Infill',    'DG',   'Bonus Unit', 'Infill', 'DG Bonus Units Infill',   forecast_residential_units_infill),
    ('PL_Bonus_MF',        'PL',   'Bonus Unit', 'MF',     'PL Bonus Units MF',       forecast_residential_units),
    ('PL_Bonus_SF',        'PL',   'Bonus Unit', 'SF',     'PL Bonus Units SF',       forecast_residential_units),
    ('PL_Bonus_Infill',    'PL',   'Bonus Unit', 'Infill', 'PL Bonus Units Infill',   forecast_residential_units_infill),
    ('WA_Bonus_MF',        'WA',   'Bonus Unit', 'MF',     'WA Bonus Units MF',       forecast_residential_units),
    ('WA_Bonus_SF',        'WA',   'Bonus Unit', 'SF',     'WA Bonus Units SF',       forecast_residential_units),
    ('WA_Bonus_Infil',     'WA',   'Bonus Unit', 'Infill', 'WA Bonus Units Infill',   forecast_residential_units_infill),
    ##-- General Unit Assignments --##
    ('CSLT_General_MF',    'CSLT', 'General', 'MF',     'CSLT General Units MF',     forecast_residential_units),
    ('CSLT_General_SF',    'CSLT', 'General', 'SF',     'CSLT General Units SF',     forecast_residential_units),
    ('CSLT_General_Infill','CSLT', 'General', 'Infill', 'CSLT General Units Infill', forecast_residential_units_infill),
    ('EL_General_MF',      'EL',   'General', 'MF',     'EL General Units MF',       forecast_residential_units),
    ('EL_General_SF',      'EL',   'General', 'SF',     'EL General Units SF',       forecast_residential_units),
    ('EL_General_Infill',  'EL',   'General', 'Infill', 'EL General Units Infill',   forecast_residential_units_infill),
    ('PL_General_MF',      'PL',   'General', 'MF',     'PL General Units MF',       forecast_residential_units),
    ('PL_General_SF',      'PL',   'General', 'SF',     'PL General Units SF',       forecast_residential_units),
    ('PL_General_Infill',  'PL',   'General', 'Infill', 'PL General Units Infill',   forecast_residential_units_infill),
    ('DG_General_MF',      'DG',   'General', 'MF',     'DG General Units MF',       forecast_residential_units),
    ('DG_General_SF',      'DG',   'General', 'SF',     'DG General Units SF',       forecast_residential_units),
    ('DG_General_Infill',  'DG',   'General', 'Infill', 'DG General Units Infill',   forecast_residential_units_infill),
    ('WA_General_MF',      'WA',   'General', 'MF',     'WA General Units MF',       forecast_residential_units),
    ('WA_General_SF',      'WA',   'General', 'SF',     'WA General Units SF',       forecast_residential_units),
    ('WA_General_Infill',  'WA',   'General', 'Infill', 'WA General Units Infill',   forecast_residential_units_infill),
]

df_built_parcels = pd.DataFrame()
for condition_key, jurisdiction, pool, unit_type, label, fn in assignments:
    target_sum             = get_target_sum(dfPool, jurisdiction, pool, unit_type)
    sdfParcels, df_summary = fn(sdfParcels, conditions[condition_key], target_sum, label)
    df_built_parcels       = pd.concat([df_built_parcels, df_summary], ignore_index=True)

df_built_parcels


> Forecast TRPA Pools

In [ ]:
## TRPA Unit Pool Assignments ##
# (condition_key, jurisdiction, pool, type, label, forecast_function)
trpa_assignments = [
    ('TRPA_Bonus_MF',     'TRPA', 'Bonus Unit', 'MF',     'TRPA Bonus Units MF',     forecast_residential_units),
    ('TRPA_Bonus_SF',     'TRPA', 'Bonus Unit', 'SF',     'TRPA Bonus Units SF',     forecast_residential_units),
    ('TRPA_Bonus_Infill', 'TRPA', 'Bonus Unit', 'Infill', 'TRPA Bonus Units Infill', forecast_residential_units_infill),
    ('TRPA_General_MF',   'TRPA', 'General',    'MF',     'TRPA General Units MF',   forecast_residential_units),
    ('TRPA_General_SF',   'TRPA', 'General',    'SF',     'TRPA General Units SF',   forecast_residential_units),
    ('TRPA_General_Infill','TRPA','General',    'Infill', 'TRPA General Units Infill',forecast_residential_units_infill),
    ('TRPA_ADU',          'TRPA', 'ADU',         'ADU',    'TRPA ADU Units',          forecast_residential_units),
]

for condition_key, jurisdiction, pool, unit_type, label, fn in trpa_assignments:
    target_sum             = get_target_sum(dfPool, jurisdiction, pool, unit_type)
    sdfParcels, df_summary = fn(sdfParcels, conditions[condition_key], target_sum, label)
    df_built_parcels       = pd.concat([df_built_parcels, df_summary], ignore_index=True)

df_built_parcels


> Assign the Remainders

In [ ]:
# use df_built_parcels to set new infill target_sum
df_remaining = df_built_parcels.loc[df_built_parcels.Total_Remaining_Units > 0]
df_remaining.Reason.to_list() 

# get first text before first space
df_remaining['Jurisdiction'] = df_remaining['Reason'].str.split(' ').str[0]
df_remaining

In [ ]:
## Infill Forecasting remaining units from other Pools ##
# (remaining_reason, condition_key, label, forecast_function)
remainder_assignments = [
    ('WA Bonus Units SF',      'WA_Bonus_Infil',     'WA Bonus Units Infill',     forecast_residential_units_infill),
    ('EL General Units MF',    'EL_General_Infill',  'EL General Units Infill',   forecast_residential_units_infill),
    ('PL General Units MF',    'PL_General_Infill',  'PL General Units Infill',   forecast_residential_units_infill),
    ('DG General Units MF',    'DG_General_Infill',  'DG General Units Infill',   forecast_residential_units),
    ('WA General Units MF',    'WA_General_Infill',  'WA General Units Infill',   forecast_residential_units_infill),
    ('WA General Units SF',    'WA_General_Infill',  'WA General Units Infill',   forecast_residential_units_infill),
    ('TRPA General Units MF',  'TRPA_General_Infill','TRPA General Units Infill', forecast_residential_units_infill),
]

for remaining_reason, condition_key, label, fn in remainder_assignments:
    target_sum             = df_remaining.loc[df_remaining.Reason == remaining_reason, 'Total_Remaining_Units'].values[0]
    sdfParcels, df_summary = fn(sdfParcels, conditions[condition_key], target_sum, label)
    df_built_parcels       = pd.concat([df_built_parcels, df_summary], ignore_index=True)

df_built_parcels


In [ ]:
# assign ADUs to existing residential parcels residential units=1 and ADU_ALLOWED = Yes
target_sum = 4385 - sdfParcels.FORECASTED_RESIDENTIAL_UNITS.sum()
sdfParcels, df_summary = forecast_residential_units(sdfParcels, TRPA_ADU_condition, target_sum, 'TRPA ADU Units')
df_built_parcels = pd.concat([df_built_parcels, df_summary], ignore_index=True)

> Check Results

In [ ]:
dfPoolMelt = dfPool.melt(id_vars=['Jurisdiction', 'Unit_Pool'], value_vars=['Future_Units_Adjusted_MF', 'Future_Units_Adjusted_SF', 'Future_Units_Adjusted_Infill'])
# drop Future_Units_Adjusted_ from variable
dfPoolMelt['variable'] = dfPoolMelt['variable'].str.replace('Future_Units_Adjusted_', '')

dfPoolMelt['Unit_Pool'] = dfPoolMelt['Jurisdiction'] + ' ' + dfPoolMelt['Unit_Pool']
dfPoolMelt.rename(columns={'variable': 'Reason', 'value': 'Units'}, inplace=True)

dfForecastGroup = sdfParcels.groupby(['FORECAST_REASON'])['FORECASTED_RESIDENTIAL_UNITS'].sum().reset_index()
# split last FORECAST Reason into two columns based on last space in string
dfForecastGroup['Reason'] = dfForecastGroup['FORECAST_REASON'].str.split(' ').str[-1]
dfForecastGroup['Jurisdiction'] = dfForecastGroup['FORECAST_REASON'].str.split(' ').str[0]
# if FORECAST_REASON valie contains 'Bonus' then set Unit Pool to Bonus Units
dfForecastGroup.loc[dfForecastGroup['FORECAST_REASON'].str.contains('Bonus'), 'Unit_Pool'] = dfForecastGroup.Jurisdiction + ' ' + 'Bonus Unit'
# if FORECAST_REASON valie contains 'General' then set Unit Pool to General Units
dfForecastGroup.loc[dfForecastGroup['FORECAST_REASON'].str.contains('General'), 'Unit_Pool'] = dfForecastGroup.Jurisdiction + ' ' + 'General'
dfMerge = dfForecastGroup.merge(dfPoolMelt, on=['Unit_Pool', 'Reason'], how='left')
dfMerge['Unit_Diff'] = dfMerge['FORECASTED_RESIDENTIAL_UNITS'] - dfMerge['Units']
dfMerge

> Assign Forecast Year

In [ ]:
## Assigning Development Year to Parcels ##
# Get total development by 2035
TotalDevelopment = dfResZoned.Future_Units.sum()
Development_2035 = (TotalDevelopment*.33).astype(int)
sdfParcels['Development_Year']=None
#Assining all development that's currently in the works as being done by 2035
sdfParcels.loc[sdfParcels['FORECAST_REASON'] == 'Assigned', 'Development_Year'] = 2035
RemainingDevelopment_2035 = Development_2035 - sdfParcels.loc[sdfParcels['FORECAST_REASON'] == 'Assigned', 'FORECASTED_RESIDENTIAL_UNITS'].sum()

Development_2035_Condition = sdfParcels['FORECASTED_RESIDENTIAL_UNITS'].where(sdfParcels['FORECAST_REASON'] != 'Assigned').cumsum()
sdfParcels.loc[Development_2035_Condition < RemainingDevelopment_2035, 'Development_Year'] = 2035
sdfParcels.loc[(sdfParcels['FORECAST_REASON']!= '') & (sdfParcels['Development_Year'].isnull()), 'Development_Year'] = 2050
development_year = sdfParcels.groupby('Development_Year')['FORECASTED_RESIDENTIAL_UNITS'].sum()
development_year

#### Assign Occupancy Rate to all forecasted residential parcels

In [ ]:
sdfParcels = pd.read_pickle(parcel_pickle_part2)
# filter to FORECAST_YEAR = 2035
#sdfParcels = sdfParcels.loc[sdfParcels['Development_Year'] == 2050]        

In [ ]:
sdfParcels['FORECASTED_RES_OCCUPANCY_RATE'] = 0
# map lookup known project occupancy rates to parcels
# Need to pull this in from the known project occupancy rates lookup
dfResAssigned = pd.read_csv("res_assigned_lookup.csv")
string_cols = dfResAssigned.select_dtypes(include=['string']).columns
dfResAssigned[string_cols] = dfResAssigned[string_cols].astype(object)
dfResAssigned['Occupancy_Rate'] = dfResAssigned['Occupancy_Rate'].astype(float)
# change all strings in dfResAssigned to be objects
string_cols=sdfParcels.select_dtypes(include=['string']).columns
sdfParcels[string_cols] = sdfParcels[string_cols].astype(object)
sdfParcels['FORECASTED_RES_OCCUPANCY_RATE'] = sdfParcels['APN'].map(dict(zip(dfResAssigned.APN, dfResAssigned['Occupancy_Rate'])))
sdfParcels.FORECAST_REASON.value_counts()
# if FORECAST_REASON has the word 'Bonus' in it set occupancy rate to 100%
sdfParcels.loc[sdfParcels['FORECAST_REASON'].fillna('').str.contains('Bonus'), 'FORECASTED_RES_OCCUPANCY_RATE'] = 1
# if FORECAST_REASON has the word 'CTC' in it set occupancy rate to 100%
sdfParcels.loc[sdfParcels['FORECAST_REASON'].fillna('').str.contains('CTC'), 'FORECASTED_RES_OCCUPANCY_RATE'] = 1
# if FORECAST_REASON has the word 'General' and housing zoning is SF_only in it set occupancy rate to 35
sdfParcels.loc[(sdfParcels['FORECAST_REASON'].fillna('').str.contains('General')), 'FORECASTED_RES_OCCUPANCY_RATE'] = 0.35

#### Assing Household Income Category - Low, Medium, High

In [ ]:
sdfParcels['FORECASTED_RES_INCOME_LOW']     = 0.0
sdfParcels['FORECASTED_RES_INCOME_MEDIUM']  = 0.0
sdfParcels['FORECASTED_RES_INCOME_HIGH']    = 0.0

# map lookup known project income levels to parcels
sdfParcels['FORECAST_RES_INCOME_CATEGORY']     = sdfParcels.APN.map(dict(zip(dfResAssigned.APN, dfResAssigned['HH Income Category'])))
# change 'Achievable' to 'Medium' and 'Affodaable to 'Low'
sdfParcels.loc[sdfParcels['FORECAST_RES_INCOME_CATEGORY'] == 'Achievable', 'FORECAST_RES_INCOME_CATEGORY'] = 'Medium'
sdfParcels.loc[sdfParcels['FORECAST_RES_INCOME_CATEGORY'] == 'Affordable', 'FORECAST_RES_INCOME_CATEGORY'] = 'Low'

# def function to if income category is low, medium, or high income field to 1
def income_category(df, category):
    df.loc[df['FORECAST_RES_INCOME_CATEGORY'] == category, 'FORECASTED_RES_INCOME_' + str(category).upper()] = 1
    return df

for category in ['Low', 'Medium', 'High']:
    sdfParcels = income_category(sdfParcels, category)

# for FORECAST_REASON with 'Bonus' set low income to 1
sdfParcels.loc[sdfParcels['FORECAST_REASON'].fillna('').str.contains('Bonus'), 'FORECASTED_RES_INCOME_LOW'] = 0.78
sdfParcels.loc[sdfParcels['FORECAST_REASON'].fillna('').str.contains('Bonus'), 'FORECASTED_RES_INCOME_MEDIUM'] = 0.2
sdfParcels.loc[sdfParcels['FORECAST_REASON'].fillna('').str.contains('Bonus'), 'FORECASTED_RES_INCOME_HIGH'] = 0.02

# for FORECAST_REASON with 'General' and housing zoning is SF_only set low income to 1
sdfParcels.loc[(sdfParcels['FORECAST_REASON'].fillna('').str.contains('General')), 'FORECASTED_RES_INCOME_LOW'] = 0.01
sdfParcels.loc[(sdfParcels['FORECAST_REASON'].fillna('').str.contains('General')), 'FORECASTED_RES_INCOME_MEDIUM'] = 0.02
sdfParcels.loc[(sdfParcels['FORECAST_REASON'].fillna('').str.contains('General')), 'FORECASTED_RES_INCOME_HIGH'] = 0.97

# assigne income levels to parcels with 'ADU' in FORECAST_REASON - UPDATED
sdfParcels.loc[sdfParcels['FORECAST_REASON'].fillna('').str.contains('ADU'), 'FORECASTED_RES_INCOME_LOW'] = 0.65
sdfParcels.loc[sdfParcels['FORECAST_REASON'].fillna('').str.contains('ADU'), 'FORECASTED_RES_INCOME_MEDIUM'] = 0.2
sdfParcels.loc[sdfParcels['FORECAST_REASON'].fillna('').str.contains('ADU'), 'FORECASTED_RES_INCOME_HIGH'] = 0.15

# for FORECAST_REASON with 'CTC' set low income to 1
sdfParcels.loc[sdfParcels['FORECAST_REASON'].fillna('').str.contains('CTC'), 'FORECASTED_RES_INCOME_LOW'] = 1



### Summary

In [ ]:
# Need to get 2035 forecasted units and 2050 forecasted units
# We need to also change the people per household to hit population targets

In [ ]:
# This should be the new base year SocioEon_Summer
socio            = 'TravelDemandModel\\2022\\data\\processed_data\\SocioEcon_Summer.csv'
socio_path       = local_path.parents[2].joinpath(socio)
dfSocio          = pd.read_csv(socio_path)

# rename columns to taz to TAZ
dfSocio.rename(columns={'taz': 'TAZ'}, inplace=True)
# group by TAZ
sdfParcels['FORECASTED_OCCUPIED_UNITS'] = sdfParcels['FORECASTED_RESIDENTIAL_UNITS'] * sdfParcels['FORECASTED_RES_OCCUPANCY_RATE']
sdfParcels['FORECASTED_RES_INCOME_LOW_UNITS']    = sdfParcels['FORECASTED_RES_INCOME_LOW'] * sdfParcels['FORECASTED_OCCUPIED_UNITS']
sdfParcels['FORECASTED_RES_INCOME_MEDIUM_UNITS'] = sdfParcels['FORECASTED_RES_INCOME_MEDIUM'] * sdfParcels['FORECASTED_OCCUPIED_UNITS']
sdfParcels['FORECASTED_RES_INCOME_HIGH_UNITS']   = sdfParcels['FORECASTED_RES_INCOME_HIGH'] * sdfParcels['FORECASTED_OCCUPIED_UNITS']

df = sdfParcels.loc[sdfParcels['FORECASTED_RESIDENTIAL_UNITS'] > 0]
dfTAZ_Summary_2035 = df.loc[df['Development_Year'] == 2035]
dfTAZ_Summary_2035 = dfTAZ_Summary_2035.groupby(['TAZ']).agg({'FORECASTED_RESIDENTIAL_UNITS': 'sum',
                                                                                    'FORECASTED_OCCUPIED_UNITS':'sum', 
                                                                                    'FORECASTED_RES_INCOME_LOW_UNITS':'sum', 'FORECASTED_RES_INCOME_MEDIUM_UNITS':'sum',
                                                                                    'FORECASTED_RES_INCOME_HIGH_UNITS':'sum'}).reset_index()

#This is total development by 2050
dfTAZ_Summary_2050 = df.groupby(['TAZ']).agg({'FORECASTED_RESIDENTIAL_UNITS': 'sum',
                                                 'FORECASTED_OCCUPIED_UNITS':'sum', 
                                                 'FORECASTED_RES_INCOME_LOW_UNITS':'sum', 'FORECASTED_RES_INCOME_MEDIUM_UNITS':'sum',
                                                 'FORECASTED_RES_INCOME_HIGH_UNITS':'sum'}).reset_index()

# merge TAZ summary with socio data

def process_dataframe(df, dfSocio):
    df = dfSocio.merge(df, on='TAZ', how='left')
    df = df.fillna(0)
    
    df['TOTAL_FORECASTED_UNITS_LOW_INCOME'] = df['FORECASTED_RES_INCOME_LOW_UNITS'] + df['occ_units_low_inc']
    df['TOTAL_FORECASTED_UNITS_MED_INCOME'] = df['FORECASTED_RES_INCOME_MEDIUM_UNITS'] + df['occ_units_med_inc']
    df['TOTAL_FORECASTED_UNITS_HIGH_INCOME'] = df['FORECASTED_RES_INCOME_HIGH_UNITS'] + df['occ_units_high_inc']
    df['TOTAL_FORECASTED_UNITS_OCCUPIED'] = df['FORECASTED_OCCUPIED_UNITS'] + df['total_occ_units']
    df['NEW_OCCUPANCY_RATE'] = (df['FORECASTED_OCCUPIED_UNITS'] + df['total_occ_units']) / (df['total_residential_units'] + df['FORECASTED_RESIDENTIAL_UNITS'])
    
    return df
dfTAZ_Summary_2035 = process_dataframe(dfTAZ_Summary_2035, dfSocio)
dfTAZ_Summary_2050 = process_dataframe(dfTAZ_Summary_2050, dfSocio)

# dfTAZ_Summary = dfTAZ_Summary.merge(dfSocio, on='TAZ', how='left')

# dfTAZ_Summary['TOTAL_FORECASTED_UNITS_LOW_INCOME'] = dfTAZ_Summary['FORECASTED_RES_INCOME_LOW_UNITS'] + dfTAZ_Summary['occ_units_low_inc']
# dfTAZ_Summary['TOTAL_FORECASTED_UNITS_MED_INCOME'] = dfTAZ_Summary['FORECASTED_RES_INCOME_MEDIUM_UNITS'] + dfTAZ_Summary['occ_units_med_inc']
# dfTAZ_Summary['TOTAL_FORECASTED_UNITS_HIGH_INCOME'] = dfTAZ_Summary['FORECASTED_RES_INCOME_HIGH_UNITS'] + dfTAZ_Summary['occ_units_high_inc']
# dfTAZ_Summary['NEW_OCCUPANCY_RATE'] = (dfTAZ_Summary['FORECASTED_OCCUPIED_UNITS'] + dfTAZ_Summary['total_occ_units'])/ (dfTAZ_Summary['total_residential_units'] + dfTAZ_Summary['FORECASTED_RESIDENTIAL_UNITS'])

# We need clarity on where these target sums came from
## Should we keep them for consistency with the RTP or let population be determined by people per household and number of occupied units

In [ ]:
target_sum_2035 = 55592
target_sum_2050 =57611

dfTAZ_Summary_2035['FORECASTED_POPULATION'] = dfTAZ_Summary_2035['TOTAL_FORECASTED_UNITS_OCCUPIED']*dfTAZ_Summary_2035['persons_per_occ_unit']
forecasted_population_2035 = dfTAZ_Summary_2035['FORECASTED_POPULATION'].sum()
dfTAZ_Summary_2050['FORECASTED_POPULATION'] = dfTAZ_Summary_2050['TOTAL_FORECASTED_UNITS_OCCUPIED']*dfTAZ_Summary_2050['persons_per_occ_unit']
forecasted_population_2050 = dfTAZ_Summary_2050['FORECASTED_POPULATION'].sum()
adjustment_factor_2035 = target_sum_2035/forecasted_population_2035
adjustment_factor_2050  = target_sum_2050/forecasted_population_2050

dfTAZ_Summary_2035['Adjusted_Persons_Per_Occupied_Unit'] = dfTAZ_Summary_2035['persons_per_occ_unit']*adjustment_factor_2035
dfTAZ_Summary_2050['Adjusted_Persons_Per_Occupied_Unit'] = dfTAZ_Summary_2050['persons_per_occ_unit']*adjustment_factor_2050

dfTAZ_Summary_2050['FORECASTED_POPULATION'] = dfTAZ_Summary_2050['TOTAL_FORECASTED_UNITS_OCCUPIED']*dfTAZ_Summary_2050['Adjusted_Persons_Per_Occupied_Unit']
dfTAZ_Summary_2035['FORECASTED_POPULATION'] = dfTAZ_Summary_2035['TOTAL_FORECASTED_UNITS_OCCUPIED']*dfTAZ_Summary_2035['Adjusted_Persons_Per_Occupied_Unit']
forecasted_population_2035 = dfTAZ_Summary_2035['FORECASTED_POPULATION'].sum()
forecasted_population_2050 = dfTAZ_Summary_2050['FORECASTED_POPULATION'].sum()


In [ ]:
print(forecasted_population_2035)
print(forecasted_population_2050)




In [ ]:
print(dfTAZ_Summary_2050['TOTAL_FORECASTED_UNITS_OCCUPIED'].sum())
print(dfTAZ_Summary_2035['TOTAL_FORECASTED_UNITS_OCCUPIED'].sum())

In [ ]:
print(dfTAZ_Summary_2035['TOTAL_FORECASTED_UNITS_MED_INCOME'].sum()+dfTAZ_Summary_2035['TOTAL_FORECASTED_UNITS_LOW_INCOME'].sum()+dfTAZ_Summary_2035['TOTAL_FORECASTED_UNITS_HIGH_INCOME'].sum())
print(dfTAZ_Summary_2050['TOTAL_FORECASTED_UNITS_MED_INCOME'].sum()+dfTAZ_Summary_2050['TOTAL_FORECASTED_UNITS_LOW_INCOME'].sum()+dfTAZ_Summary_2050['TOTAL_FORECASTED_UNITS_HIGH_INCOME'].sum())

In [ ]:
# save to pickle
taz_summary_2035_pickle = data_dir / 'taz_summary_2035.pickle'
taz_summary_2050_pickle = data_dir / 'taz_summary_2050.pickle'
dfTAZ_Summary_2035.to_pickle(taz_summary_2035_pickle)
dfTAZ_Summary_2035.to_csv(data_dir / 'taz_summary_2035.csv')
dfTAZ_Summary_2050.to_pickle(taz_summary_2050_pickle)
dfTAZ_Summary_2050.to_csv(data_dir / 'taz_summary_2050.csv')

##### QA Extra

In [ ]:
built_units = sdfParcels.groupby('FORECAST_REASON').agg({'FORECASTED_RESIDENTIAL_UNITS':'sum'})
dfResZoned = dfPool.copy()
# Create a dictionary to map the forecast reason to the Jurisdiction and Unit Pool
Forecast_Reason_lookup = {'CSLT Bonus Units Built':{'Jurisdiction':'CSLT', 'Unit_Pool':'Bonus Unit'},
                          'CSLT General Units Built':{'Jurisdiction':'CSLT', 'Unit_Pool':'General'},
                          'EL General Units Built':{'Jurisdiction':'EL', 'Unit_Pool':'General'},
                          'PL Bonus Units Built':{'Jurisdiction':'PL', 'Unit_Pool':'Bonus Unit'},
                          'PL General Units Built':{'Jurisdiction':'PL', 'Unit_Pool':'General'},
                          'WA Bonus Units Built':{'Jurisdiction':'WA', 'Unit_Pool':'Bonus Unit'},
                          'WA General Units Built':{'Jurisdiction':'WA', 'Unit_Pool':'General'},
                          'DG Bonus Units Built':{'Jurisdiction':'DG', 'Unit_Pool':'Bonus Unit'},
                          'DG General Units Built':{'Jurisdiction':'DG', 'Unit_Pool':'General'},
                          'TRPA Bonus Units Built':{'Jurisdiction':'TRPA', 'Unit_Pool':'Bonus Unit'},
                          'TRPA General Units Built':{'Jurisdiction':'TRPA', 'Unit_Pool':'General'},
                          'ADU Units Built':{'Jurisdiction':'TRPA', 'Unit_Pool':'ADU'}}
# Map 'Jurisdiction' and 'Unit_Pool' separately from the dictionary
built_units['Jurisdiction'] = built_units.index.map(lambda x: Forecast_Reason_lookup.get(x, {}).get('Jurisdiction'))
built_units['Unit_Pool'] = built_units.index.map(lambda x: Forecast_Reason_lookup.get(x, {}).get('Unit_Pool'))
unit_comparison = built_units.merge(dfResZoned, how='left', on=['Jurisdiction', 'Unit_Pool'])
unit_comparison['Difference'] = unit_comparison['Future_Units_Adjusted'] - unit_comparison['FORECASTED_RESIDENTIAL_UNITS']
unit_comparison.to_csv('unit_comparison.csv', index=False)

In [ ]:
# forecasted residential uints by location to twon center
sdfParcels.groupby(['LOCATION_TO_TOWNCENTER','FORECAST_REASON'])['FORECASTED_RESIDENTIAL_UNITS'].sum()

In [ ]:
# gropu by TAZ and sum forecasted residential units and residential units
# Forecast year total residential units by TAZ
sdfParcels.groupby('TAZ')[['FORECASTED_RESIDENTIAL_UNITS', 'Residential_Units']].sum().reset_index()
# total units with forecast year 2035 and 2050
sdfParcels.groupby('Development_Year')[['FORECASTED_RESIDENTIAL_UNITS', 'Residential_Units']].sum().reset_index()

### Tourist Accommodation Forecast

In [ ]:
# lookup lists
tau_assigned_lookup = "Lookup_Lists/forecast_tourist_assigned_units.csv"
# get assigned units lookup as data frames
dfTouristAssigned = pd.read_csv(tau_assigned_lookup)
# assign tourist units to parcels
sdfParcels['FORECASTED_TOURIST_UNITS'] = 0
# set tourist units to assigned total
sdfParcels['FORECASTED_TOURIST_UNITS'] = sdfParcels.APN.map(dict(zip(dfTouristAssigned.APN, dfTouristAssigned['Unit_Pool'])))

### Commercial Floor Area Forecast

In [ ]:
# lookup lists
cfa_assigned_lookup = "Lookup_Lists/forecast_commercial_assigned_units.csv"
# get zoned units lookups as data frames
dfCommercialAssigned = pd.read_csv(cfa_assigned_lookup)
# set commercial floor area to assigned total
sdfParcels['FORECASTED_COMMERCIAL_FLOOR_AREA'] = 0
sdfParcels['FORECASTED_COMMERCIAL_FLOOR_AREA'] = sdfParcels.APN.map(dict(zip(dfCommercialAssigned.APN, dfCommercialAssigned.SqFt)))

### Exports

In [ ]:
# export to pickle
sdfParcels.to_pickle(parcel_pickle_part2)
# to csv
sdfParcels.to_csv(data_dir/'Parcels_Forecast.csv', index=False)
# to feature class
sdfParcels.spatial.to_featureclass(Path(gdb)/'Parcel_Forecast')

# Summarize Existing and Forecasted Units by Jurisdiction and Unit Pool by TAZ
dfTAZ = sdfParcels.groupby('TAZ')[['FORECASTED_RESIDENTIAL_UNITS', 'Residential_Units']].sum().reset_index()
dfTAZ.to_csv(data_dir/'TAZ_Units.csv', index=False)